# Domain-adapting MobileCLIP2-S2 on Pl@ntNet-300K

**Runtime → Change runtime type → A100.** Roughly 2–3 hours end to end:
~30 min to fetch 31.7 GB, ~90 min to train, minutes to save.

## What this is testing

The 17.9 MB encoder scores **0.6236** species top-1 on the 490-class catalogue.
`plantclef24` at 43 MB scores **0.7671**. Three inference-side levers were measured
and all three are null (`SMALL_FRONTIER_FINDINGS.md`), so that 14.4pp is in the
encoder's representation.

What `plantclef24` *is*, is a DINOv2 ViT-B fine-tuned on 7,806 Pl@ntNet species —
stock DINOv2 ViT-B is unremarkable on this task. **So the 43 MB advantage is domain
adaptation, not size**, and nobody has done that to a 17.9 MB model.

This notebook runs the same recipe one encoder-scale down. It trains the image tower
with a plain classification objective, throws the classifier away, and saves the
frozen tower. Evaluation happens back on your machine through the normal path.

Predictions are recorded in `ADAPT_PREREG.md` **before** this was run: adapted-S2
lands between 0.65 and 0.72, genus moves more than species, and 0.74+ would change
the size decision.

> **What it cannot show.** 530 of the 1,081 training species *are* the catalogue,
> because the catalogue was selected by image availability and absorbed 299,832 of
> the 306,146 images. So this produces a catalogue-flavoured encoder, and the
> standard evaluation cannot tell that apart from a general plant encoder. The final
> cell runs the transfer probe that can.


## 1. Environment


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install open_clip_torch==2.32.0 pandas pyarrow


In [ ]:
import os, json, time, zipfile, pathlib, random
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image

DEV = 'cuda'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(torch.__version__, torch.cuda.get_device_name(0))


## 2. The research repo

Cloned for the metadata and so the recipe sits next to the findings it came from.
Nothing here needs the private `data/` directory — the images come from Zenodo.


In [ ]:
!git clone -q https://github.com/semajyllek/narrowcast-plantid.git
REPO = pathlib.Path('narrowcast-plantid')
print(open(REPO / 'ADAPT_PREREG.md').read()[:1200])


## 3. Pl@ntNet-300K

31.7 GB, 306,146 images, 1,081 species. The download is the slow part; the extract
is quick. If Colab drops the connection, re-run — `curl -C -` resumes.


In [ ]:
import subprocess
ZIP = pathlib.Path('/content/plantnet_300K.zip')
URL = 'https://zenodo.org/api/records/5645731/files/plantnet_300K.zip/content'
# subprocess rather than a shell escape: `!` inside an if-block is IPython-only
# and this notebook should also run under plain jupyter.
if not (ZIP.exists() and ZIP.stat().st_size > 31e9):
    subprocess.run(['curl', '-L', '-C', '-', '--retry', '5', '-o', str(ZIP), URL],
                   check=True)
print(f'{ZIP.stat().st_size/1e9:.1f} GB')


In [ ]:
ROOT = pathlib.Path('/content/plantnet')
if not ROOT.exists():
    t0 = time.time()
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(ROOT)
    print(f'extracted in {time.time()-t0:.0f}s')
IMAGES = next(p for p in ROOT.rglob('images') if p.is_dir())
print(IMAGES, len(list(IMAGES.iterdir())), 'split dirs or species dirs')


## 4. The training set

Pl@ntNet-300K ships its own train/val/test split and we keep it: `val` is used to
watch for overfitting and nothing here is selected on `test`.

Labels are the 1,081 species ids. The classifier is thrown away afterwards — only
the tower is kept — so the head width is irrelevant to the product.


In [ ]:
# Derived from the directory structure -- images/<split>/<species_id>/<id>.jpg --
# rather than from the metadata json, which the archive does not always carry.
rows = [(p.stem, p.parent.name, p.parent.parent.name, p)
        for p in IMAGES.rglob('*.jpg')]
df = pd.DataFrame(rows, columns=['image_id', 'species_id', 'split', 'path'])
assert set(df['split']) <= {'train', 'val', 'test'}, sorted(set(df['split']))[:6]

classes = sorted(df['species_id'].unique())
cls_idx = {c: i for i, c in enumerate(classes)}
df['y'] = df['species_id'].map(cls_idx)
print(f"{len(df):,} images, {len(classes)} species")
print(df['split'].value_counts().to_dict())


## 5. Model

`create_model_and_transforms` gives the CLIP pair; only `.visual` is trained, and a
linear classifier is bolted on for the objective. The text tower is dropped — it
never ships and training it would waste the GPU.


In [ ]:
import open_clip
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    'hf-hub:timm/MobileCLIP2-S2-OpenCLIP')
tower = model.visual.to(DEV)
del model

with torch.no_grad():
    d = tower(torch.zeros(1, 3, 256, 256, device=DEV)).shape[-1]
print('embedding dim', d, '| params',
      f'{sum(p.numel() for p in tower.parameters())/1e6:.1f}M')

head = nn.Linear(d, len(classes)).to(DEV)


## 6. Data pipeline

`preprocess_train` is open_clip's own augmentation for this model, so the adapted
tower stays consistent with how it was pretrained.


In [ ]:
class PN(Dataset):
    def __init__(self, frame, tf):
        self.p = frame['path'].tolist(); self.y = frame['y'].tolist(); self.tf = tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        try:
            im = Image.open(self.p[i]).convert('RGB')
        except Exception:
            im = Image.new('RGB', (256, 256))
        return self.tf(im), self.y[i]

tr = DataLoader(PN(df[df.split=='train'], preprocess_train), batch_size=256,
                shuffle=True, num_workers=8, pin_memory=True, drop_last=True,
                persistent_workers=True)
va = DataLoader(PN(df[df.split=='val'], preprocess_val), batch_size=512,
                shuffle=False, num_workers=8, pin_memory=True)
print(len(tr), 'train batches |', len(va), 'val batches')


## 7. Train

Plain cross-entropy, AdamW, cosine schedule, bf16 autocast. The tower gets a lower
learning rate than the fresh head — standard for fine-tuning a pretrained backbone
under a randomly initialised classifier, which would otherwise wreck the features
in the first few hundred steps.

**~90 min for 4 epochs on an A100.** Drop `EPOCHS` to 1 for a smoke test first.


In [ ]:
EPOCHS = 4
opt = torch.optim.AdamW([
    {'params': tower.parameters(), 'lr': 1e-4},
    {'params': head.parameters(),  'lr': 1e-3},
], weight_decay=0.05)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=[1e-4, 1e-3], total_steps=EPOCHS*len(tr), pct_start=0.1)

def evaluate():
    tower.eval(); head.eval(); ok = n = 0
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        for x, y in va:
            p = head(tower(x.to(DEV, non_blocking=True))).argmax(1).cpu()
            ok += (p == y).sum().item(); n += len(y)
    tower.train(); head.train()
    return ok / max(n, 1)

print(f'val top-1 before adaptation: {evaluate():.4f}')
t0 = time.time()
for ep in range(EPOCHS):
    tower.train(); head.train()
    for i, (x, y) in enumerate(tr):
        x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss = F.cross_entropy(head(tower(x)), y, label_smoothing=0.1)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if i % 100 == 0:
            print(f'  ep{ep} {i}/{len(tr)} loss {loss.item():.3f} '
                  f'({time.time()-t0:.0f}s)', flush=True)
    print(f'epoch {ep}: val top-1 {evaluate():.4f}  ({time.time()-t0:.0f}s)', flush=True)


## 8. Save the tower

The classifier is discarded here — the product is a frozen feature extractor, and
keeping a 1,081-way head would only invite someone to use it.


In [ ]:
OUT = pathlib.Path('/content/mobileclip2_s2_plantnet.pt')
torch.save({k: v.cpu() for k, v in tower.state_dict().items()}, OUT)
print(f'{OUT} — {OUT.stat().st_size/1e6:.1f} MB')

from google.colab import files
files.download(str(OUT))


## 9. The transfer probe the pre-registration requires

530 of the 1,081 training species **are** the catalogue. So a good score on the
standard evaluation cannot distinguish *learned plants* from *learned this label
set*. This probes the 551 species the catalogue left behind — about nine images
each, too thin to train on, enough to test on.

A linear probe on held-out species, stock tower against adapted. If the adapted
tower does **not** beat stock here, it learned the catalogue rather than plants,
and any claim that it is a general small plant encoder is unsupported.


In [ ]:
# The catalogue's species ids, committed to the repo precisely so this cell can run:
# data/ is gitignored, so without it the public repo cannot say what the catalogue is.
cat_ids = set(json.load(open(REPO / 'catalogue_species.json'))['species'])
heldout = df[~df.species_id.isin(cat_ids)]
print(f'catalogue: {len(cat_ids)} species')
print(f'held out : {heldout.species_id.nunique()} species, {len(heldout):,} images')


In [ ]:
def embed_all(tw, frame):
    dl = DataLoader(PN(frame, preprocess_val), batch_size=512, num_workers=8)
    tw.eval(); out = []
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        for x, _ in dl:
            f = tw(x.to(DEV)).float()
            out.append(F.normalize(f, dim=-1).cpu().numpy())
    return np.vstack(out)

def probe(X, y, seed=0):
    from sklearn.linear_model import LogisticRegression
    rng = np.random.default_rng(seed); idx = rng.permutation(len(X))
    cut = int(0.6 * len(X)); tr_i, te_i = idx[:cut], idx[cut:]
    clf = LogisticRegression(max_iter=2000, C=10.0).fit(X[tr_i], y[tr_i])
    return float((clf.predict(X[te_i]) == y[te_i]).mean())

sub = heldout.groupby('species_id').filter(lambda g: len(g) >= 6)
y = sub['species_id'].astype('category').cat.codes.to_numpy()
print(f'probing {len(sub):,} images over {sub.species_id.nunique()} species')

stock, _, _ = open_clip.create_model_and_transforms('hf-hub:timm/MobileCLIP2-S2-OpenCLIP')
a = probe(embed_all(stock.visual.to(DEV), sub), y)
b = probe(embed_all(tower, sub), y)
print(f'held-out species linear probe — stock {a:.4f} | adapted {b:.4f} | {b-a:+.4f}')
print('Adapted must beat stock here. If it does not, it learned the catalogue'
      ' rather than plants, and no general-encoder claim is supported.')


## 10. Back on your machine

```bash
mkdir -p data/processed/adapted
mv ~/Downloads/mobileclip2_s2_plantnet.pt data/processed/adapted/

# embed the three corpora with the adapted tower
PYTHONPATH=. .venv-mps/bin/python -c "
from plantid.features import embed_catalog, embed_inat, embed_background
for f in (embed_catalog.main, embed_inat.main, embed_background.main):
    f('mobileclip2_s2_ft')"

# the standard evaluation, unchanged
PYTHONPATH=. .venv/bin/python -m plantid.eval.rejection --variant mobileclip2_s2_ft
```

Compare against `mobileclip2_s2` 0.6236 / 0.8122, `plantclef24` 0.7671 / 0.9258,
and `bioclip2_cml4` 0.8370 / 0.9735. Send me the numbers and I'll write it up
against the predictions.
